In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_curve, auc, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
DATA_DIR = "data"

dataset_paths = {
    "dns_testing": f"{DATA_DIR}/DNS-testing.parquet",
    "ldap_testing": f"{DATA_DIR}/LDAP-testing.parquet",
    "ldap_training": f"{DATA_DIR}/LDAP-training.parquet",
    "mssql_testing": f"{DATA_DIR}/MSSQL-testing.parquet",
    "mssql_training": f"{DATA_DIR}/MSSQL-training.parquet",
    "ntp_testing": f"{DATA_DIR}/NTP-testing.parquet",
    "netbios_training": f"{DATA_DIR}/NetBIOS-training.parquet",
    "netbios_testing": f"{DATA_DIR}/NetBIOS-testing.parquet",
    "portmap_training": f"{DATA_DIR}/Portmap-training.parquet",
    "snmp_testing": f"{DATA_DIR}/SNMP-testing.parquet",
    "syn_testing": f"{DATA_DIR}/Syn-testing.parquet",
    "syn_training": f"{DATA_DIR}/Syn-training.parquet",
    "tftp_testing": f"{DATA_DIR}/TFTP-testing.parquet",
    "udp_testing": f"{DATA_DIR}/UDP-testing.parquet",
    "udp_training": f"{DATA_DIR}/UDP-training.parquet",
    "udplag_testing": f"{DATA_DIR}/UDPLag-testing.parquet",
    "udplag_training": f"{DATA_DIR}/UDPLag-training.parquet",
}

In [3]:
print("Loading datasets...")
dataframes = {name: pd.read_parquet(p) for name, p in dataset_paths.items()}
dataframes_list = list(dataframes.items())
print(f"Loaded {len(dataframes)} files")

Loading datasets...
Loaded 17 files


In [4]:
label_mapping = {
    'LDAP': 'DrDoS_LDAP',
    'MSSQL': 'DrDoS_MSSQL',
    'NetBIOS': 'DrDoS_NetBIOS',
    'UDP-lag': 'DDoS_UDP_Lag',
    'UDPLag': 'DDoS_UDP_Lag',
    'UDP': 'DrDoS_UDP',        # FIXED: was 'DDoS_UDP', caused duplicate class
    'Syn': 'DDoS_SYN',
    'Portmap': 'DrDoS_Portmap',
    'TFTP': 'DrDoS_TFTP',
    'WebDDoS': 'DDoS_Web'
}

for name, df in dataframes.items():
    df['Label'] = df['Label'].map(lambda x: label_mapping.get(x, x))

In [5]:
combined_datasets = {}
for name, df in dataframes_list:
    base_name = name.split('_')[0]
    if base_name not in combined_datasets:
        combined_datasets[base_name] = df
    else:
        combined_datasets[base_name] = pd.concat([combined_datasets[base_name], df], ignore_index=True)

df = pd.concat(combined_datasets.values(), ignore_index=True)
print(f"Combined shape: {df.shape}")

Combined shape: (431371, 78)


In [6]:
print(f"Number of unique labels: {df['Label'].nunique()}")
print(df['Label'].value_counts())

Number of unique labels: 13
Label
DrDoS_NTP        121368
DrDoS_TFTP        98917
Benign            97831
DDoS_SYN          49373
DrDoS_UDP         28510
DrDoS_MSSQL       14735
DDoS_UDP_Lag       8927
DrDoS_DNS          3669
DrDoS_LDAP         3346
DrDoS_SNMP         2717
DrDoS_NetBIOS      1242
DrDoS_Portmap       685
DDoS_Web             51
Name: count, dtype: int64


In [7]:
# Check for infinities and NaNs before cleaning
print("Infinite values per column (top 10):")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sort_values(ascending=False).head(10))

print(f"\nTotal NaNs: {df.isna().sum().sum()}")
print(f"Total duplicate rows: {df.duplicated().sum()}")

Infinite values per column (top 10):
Protocol                    0
Flow Duration               0
Total Fwd Packets           0
Total Backward Packets      0
Fwd Packets Length Total    0
Bwd Packets Length Total    0
Fwd Packet Length Max       0
Fwd Packet Length Min       0
Fwd Packet Length Mean      0
Fwd Packet Length Std       0
dtype: int64

Total NaNs: 0
Total duplicate rows: 9258


In [8]:
# Drop duplicates first — before splitting, so no duplicate row ends up in both train and test
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}")

# Identify non-numeric / identifier columns that shouldn't be used as model features
non_feature_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNon-numeric columns (excluded from features): {non_feature_cols}")

# Feature columns = everything numeric, excluding the label
feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumber of numeric feature columns: {len(feature_cols)}")
print(feature_cols)

Shape after removing duplicates: (422113, 78)

Non-numeric columns (excluded from features): ['Label']

Number of numeric feature columns: 77
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Cou

In [9]:
from sklearn.preprocessing import LabelEncoder

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['Label'])

print("Label classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

X = df[feature_cols].values

# Split BEFORE scaling — this is the fix for the leakage issue
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTrain shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Label classes: ['Benign' 'DDoS_SYN' 'DDoS_UDP_Lag' 'DDoS_Web' 'DrDoS_DNS' 'DrDoS_LDAP'
 'DrDoS_MSSQL' 'DrDoS_NTP' 'DrDoS_NetBIOS' 'DrDoS_Portmap' 'DrDoS_SNMP'
 'DrDoS_TFTP' 'DrDoS_UDP']
Number of classes: 13

Train shape: (316584, 77)
Test shape: (105529, 77)


In [10]:
scaler = MinMaxScaler()

# Fit only on training data — test data must never influence scaler statistics
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the scaler fitted on train (no fitting here)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_train_scaled range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"X_test_scaled range: [{X_test_scaled.min():.4f}, {X_test_scaled.max():.4f}]")

Scaling complete.
X_train_scaled range: [0.0000, 1.0000]
X_test_scaled range: [0.0000, 4.5321]


In [11]:
benign_label_idx = list(label_encoder.classes_).index('Benign')
print(f"Benign label index: {benign_label_idx}")

# Autoencoder trains only on benign traffic from the TRAINING set
benign_mask_train = (y_train == benign_label_idx)
X_train_benign = X_train_scaled[benign_mask_train]

print(f"Benign training samples: {X_train_benign.shape[0]}")

# Reshape for LSTM: (samples, timesteps, features) — using 1 timestep since these are flow-level, not sequence, features
X_train_benign_lstm = X_train_benign.reshape((X_train_benign.shape[0], 1, X_train_benign.shape[1]))
X_test_scaled_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

print(f"LSTM input shape (train benign): {X_train_benign_lstm.shape}")
print(f"LSTM input shape (test, all classes): {X_test_scaled_lstm.shape}")

Benign label index: 0
Benign training samples: 71028
LSTM input shape (train benign): (71028, 1, 77)
LSTM input shape (test, all classes): (105529, 1, 77)


In [12]:
n_features = X_train_benign_lstm.shape[2]

# Encoder
input_layer = Input(shape=(1, n_features))
encoded = LSTM(64, activation='relu', return_sequences=True,
               kernel_regularizer=regularizers.l2(1e-4))(input_layer)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.2)(encoded)
encoded = LSTM(32, activation='relu', return_sequences=False)(encoded)

# Bottleneck -> Decoder
decoded = tf.keras.layers.RepeatVector(1)(encoded)
decoded = LSTM(32, activation='relu', return_sequences=True)(decoded)
decoded = BatchNormalization()(decoded)
decoded = Dropout(0.2)(decoded)
decoded = LSTM(64, activation='relu', return_sequences=True)(decoded)
decoded = tf.keras.layers.TimeDistributed(Dense(n_features))(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 77)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 1, 64)          │        36,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 1, 32)          │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 1, 64)          │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 77)          │         5,005 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,309 (341.05 KB)

 Trainable params: 87,117 (340.30 KB)

 Non-trainable params: 192 (768.00 B)

In [13]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
checkpoint = ModelCheckpoint('models/autoencoder_model.keras', monitor='val_loss', save_best_only=True)

history = autoencoder.fit(
    X_train_benign_lstm, X_train_benign_lstm,
    epochs=50,
    batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0157 - val_loss: 0.0339 - learning_rate: 0.0010
Epoch 2/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0025 - val_loss: 0.0164 - learning_rate: 0.0010
Epoch 3/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0017 - val_loss: 0.0018 - learning_rate: 0.0010
Epoch 4/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0012 - val_loss: 6.7152e-04 - learning_rate: 0.0010
Epoch 5/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 9.6357e-04 - val_loss: 5.6359e-04 - learning_rate: 0.0010
Epoch 6/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 8.2714e-04 - val_loss: 5.1361e-04 - learning_rate: 0.0010
Epoch 7/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.3580e-04 - val_loss: 5.5353e-04 - learning_rate: 0.0010
Epoch 8/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.7458e-04 - val_loss: 4.4324e-04 - learning_rate: 0.0010
Epoch 9/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 6.3301e-04 - val_loss: 4

In [14]:
# Re-create the same validation split Keras used internally (validation_split=0.15, no shuffle by default)
val_split_idx = int(len(X_train_benign_lstm) * 0.85)
X_val_benign_lstm = X_train_benign_lstm[val_split_idx:]

print(f"Validation benign samples: {X_val_benign_lstm.shape[0]}")

# Reconstruct validation benign samples
val_reconstructions = autoencoder.predict(X_val_benign_lstm, verbose=0)

# Reconstruction error = mean squared error per sample
val_reconstruction_errors = np.mean(np.square(X_val_benign_lstm - val_reconstructions), axis=(1, 2))

print(f"Validation reconstruction error stats:")
print(f"  Mean: {val_reconstruction_errors.mean():.6f}")
print(f"  Std:  {val_reconstruction_errors.std():.6f}")
print(f"  Min:  {val_reconstruction_errors.min():.6f}")
print(f"  Max:  {val_reconstruction_errors.max():.6f}")

# Threshold = 95th percentile of benign reconstruction error on VALIDATION data (not test)
threshold = np.percentile(val_reconstruction_errors, 95)
print(f"\nDerived anomaly threshold (95th percentile, validation set): {threshold:.6f}")

Validation benign samples: 10655
Validation reconstruction error stats:
  Mean: 0.000210
  Std:  0.000870
  Min:  0.000006
  Max:  0.023906

Derived anomaly threshold (95th percentile, validation set): 0.000715


In [15]:
# Reconstruct entire test set
test_reconstructions = autoencoder.predict(X_test_scaled_lstm, verbose=0)
test_reconstruction_errors = np.mean(np.square(X_test_scaled_lstm - test_reconstructions), axis=(1, 2))

# Binary ground truth: is this row actually an attack (not benign)?
y_test_binary = (y_test != benign_label_idx).astype(int)

# Binary prediction: does reconstruction error exceed threshold?
y_pred_binary = (test_reconstruction_errors > threshold).astype(int)

print("=== Anomaly Detection (Benign vs Attack) — TEST SET ===")
print(f"Accuracy:  {accuracy_score(y_test_binary, y_pred_binary):.4f}")
print(f"Precision: {precision_score(y_test_binary, y_pred_binary):.4f}")
print(f"Recall:    {recall_score(y_test_binary, y_pred_binary):.4f}")
print(f"F1 Score:  {f1_score(y_test_binary, y_pred_binary):.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test_binary, y_pred_binary)
print(cm)
print("(rows = actual [Benign, Attack], cols = predicted [Benign, Attack])")

=== Anomaly Detection (Benign vs Attack) — TEST SET ===
Accuracy:  0.9092
Precision: 0.9844
Recall:    0.8972
F1 Score:  0.9388

Confusion Matrix:
[[22512  1164]
 [ 8415 73438]]
(rows = actual [Benign, Attack], cols = predicted [Benign, Attack])


In [16]:
# Classifier trains on ATTACK rows only (excluding benign), from train and test sets
attack_mask_train = (y_train != benign_label_idx)
attack_mask_test = (y_test != benign_label_idx)

X_train_attacks = X_train_scaled[attack_mask_train]
y_train_attacks = y_train[attack_mask_train]

X_test_attacks = X_test_scaled[attack_mask_test]
y_test_attacks = y_test[attack_mask_test]

print(f"Train attack samples: {X_train_attacks.shape[0]}")
print(f"Test attack samples: {X_test_attacks.shape[0]}")

# Compute capped class weights — raw inverse-frequency would overweight DDoS_Web (n=51) too aggressively
from sklearn.utils.class_weight import compute_class_weight

unique_classes = np.unique(y_train_attacks)
raw_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_attacks)

MAX_WEIGHT_CAP = 15.0  # prevents any single rare class from dominating the loss
capped_weights = np.minimum(raw_weights, MAX_WEIGHT_CAP)

class_weight_dict = dict(zip(unique_classes, capped_weights))

print("\nClass weights (capped at 15.0):")
for cls_idx, weight in class_weight_dict.items():
    cls_name = label_encoder.classes_[cls_idx]
    print(f"  {cls_name}: {weight:.2f}")

Train attack samples: 245556
Test attack samples: 81853

Class weights (capped at 15.0):
  DDoS_SYN: 0.57
  DDoS_UDP_Lag: 3.06
  DDoS_Web: 15.00
  DrDoS_DNS: 7.44
  DrDoS_LDAP: 10.03
  DrDoS_MSSQL: 2.23
  DrDoS_NTP: 0.22
  DrDoS_NetBIOS: 15.00
  DrDoS_Portmap: 15.00
  DrDoS_SNMP: 10.04
  DrDoS_TFTP: 0.28
  DrDoS_UDP: 0.99


In [17]:
n_classes = len(label_encoder.classes_)

classifier = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,), kernel_regularizer=regularizers.l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])

classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
classifier.summary()

d:\CODE\PROJECTS 2026\RealTime-DDoS-Detector\training\venv_training\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 128)            │         9,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 13)             │           429 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,517 (84.05 KB)

 Trainable params: 21,133 (82.55 KB)

 Non-trainable params: 384 (1.50 KB)

In [18]:
early_stop_clf = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint_clf = ModelCheckpoint('models/classifier_model.keras', monitor='val_loss', save_best_only=True)

history_clf = classifier.fit(
    X_train_attacks, y_train_attacks,
    epochs=50,
    batch_size=256,
    validation_split=0.15,
    class_weight=class_weight_dict,
    callbacks=[early_stop_clf, checkpoint_clf],
    verbose=1
)

Epoch 1/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8332 - loss: 0.9370 - val_accuracy: 0.8813 - val_loss: 0.3812
Epoch 2/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9073 - loss: 0.6844 - val_accuracy: 0.8368 - val_loss: 0.5480
Epoch 3/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9153 - loss: 0.6508 - val_accuracy: 0.9224 - val_loss: 0.2639
Epoch 4/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9174 - loss: 0.6346 - val_accuracy: 0.8867 - val_loss: 0.3792
Epoch 5/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9203 - loss: 0.6257 - val_accuracy: 0.9313 - val_loss: 0.2326
Epoch 6/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9220 - loss: 0.6140 - val_accuracy: 0.9289 - val_loss: 0.2571
Epoch 7/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9244 - loss: 0.6081 - val_accuracy: 0.8770 - val_loss: 0.3600
Epoch 8/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9356 - loss: 0.5919 - val_accuracy: 0.

In [19]:
# Predict on test attack samples
y_pred_probs = classifier.predict(X_test_attacks, verbose=0)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Get class names for the report (only attack classes are present here, Benign excluded)
attack_class_indices = np.unique(y_test_attacks)
attack_class_names = [label_encoder.classes_[i] for i in attack_class_indices]

print("=== Classifier Performance — TEST SET (per-class) ===\n")
print(classification_report(
    y_test_attacks, y_pred_classes,
    labels=attack_class_indices,
    target_names=attack_class_names,
    zero_division=0
))

print("\nOverall weighted accuracy:", accuracy_score(y_test_attacks, y_pred_classes))

=== Classifier Performance — TEST SET (per-class) ===

               precision    recall  f1-score   support

     DDoS_SYN       1.00      0.98      0.99     11945
 DDoS_UDP_Lag       0.89      0.73      0.80      2229
     DDoS_Web       0.12      0.31      0.18        13
    DrDoS_DNS       0.48      0.38      0.43       917
   DrDoS_LDAP       0.37      0.55      0.44       681
  DrDoS_MSSQL       0.80      0.90      0.85      3057
    DrDoS_NTP       1.00      0.99      0.99     30342
DrDoS_NetBIOS       0.39      0.35      0.37       208
DrDoS_Portmap       0.27      0.63      0.38       171
   DrDoS_SNMP       0.38      0.36      0.37       679
   DrDoS_TFTP       1.00      0.99      0.99     24730
    DrDoS_UDP       0.91      0.94      0.93      6881

     accuracy                           0.96     81853
    macro avg       0.63      0.68      0.64     81853
 weighted avg       0.96      0.96      0.96     81853


Overall weighted accuracy: 0.956250839920345


In [20]:
import os
os.makedirs('models', exist_ok=True)

# Save the scaler (fitted on train only — needed to preprocess any new incoming data identically)
joblib.dump(scaler, 'models/scaler.joblib')

# Save the label encoder (maps class indices back to human-readable attack names)
joblib.dump(label_encoder, 'models/label_encoder.joblib')

# Save the exact feature column list and order — critical, since ml_service.py
# must build incoming traffic vectors in this exact same column order
joblib.dump(feature_cols, 'models/feature_cols.joblib')

# Save the derived anomaly threshold (not hardcoded — from validation set)
joblib.dump(threshold, 'models/anomaly_threshold.joblib')

print("Saved artifacts:")
for f in os.listdir('models'):
    print(f"  models/{f}")

Saved artifacts:
  models/anomaly_threshold.joblib
  models/autoencoder_model.keras
  models/classifier_model.keras
  models/feature_cols.joblib
  models/label_encoder.joblib
  models/scaler.joblib
